*Procesamiento de Lenguaje Natural*  
*Tarea 2* - Clasificador de Texto por categorias de Videos  
*Autores:* González Daniel, Guerra Michael  
*Fecha:* 24/09/2025

# Importar Librerias

In [26]:
import pandas as pd
import numpy as np

# Carga de Database

In [27]:
df = pd.read_csv('DB/youtube.csv')

print(df.count())
display(df.head())

link           3599
title          3599
description    3599
category       3599
dtype: int64


,link,title,description,category
0,JLZlCZ0,Ep 1| Travelling through North East India | Of...,Tanya Khanijow\r\n671K subscribers\r\nSUBSCRIB...,travel
1,i9E_Blai8vk,Welcome to Bali | Travel Vlog | Priscilla Lee,Priscilla Lee\r\n45.6K subscribers\r\nSUBSCRIB...,travel
2,r284c-q8oY,My Solo Trip to ALASKA | Cruising From Vancouv...,Allison Anderson\r\n588K subscribers\r\nSUBSCR...,travel
3,Qmi-Xwq-ME,Traveling to the Happiest Country in the World!!,Yes Theory\r\n6.65M subscribers\r\nSUBSCRIBE\r...,travel
4,_lcOX55Ef70,Solo in Paro Bhutan | Tiger's Nest visit | Bhu...,Tanya Khanijow\r\n671K subscribers\r\nSUBSCRIB...,travel


# Analisis Exploratorio

En este caso no se usara la columna link debido a que no nos aporta información relevante.

In [28]:
df.drop(columns=['link'], inplace=True)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3599 entries, 0 to 3598
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        3599 non-null   object
 1   description  3599 non-null   object
 2   category     3599 non-null   object
dtypes: object(3)
memory usage: 84.5+ KB


Podemos observar que no tenemos valores nulos en nuestra DB.

## Limpieza

Quitamos los valores duplicados para tener variabilidad y no varias veces el mismo valor.

In [30]:
df.drop_duplicates(inplace=True)

In [31]:
df.count()

title          3433
description    3433
category       3433
dtype: int64

Vemos que nuestro DB diminuyo un poco pero seguimos teniendo buena cantidad de datos.

Una vez hecho esto nos aseguramos de que los datos si sean strings y elimina los espacios en blanco al inicio y al final de cada título y en la descripcion.

In [32]:
df["title"] = df["title"].astype(str).str.strip()
df["description"] = df["description"].astype(str).fillna("").str.strip()

Nos aseguramos que tengamos las categorias esperadas para clasificar.

In [33]:
print("Categorías únicas:")
print(df['category'].unique())

Categorías únicas:
['travel' 'food' 'art_music' 'history']


Por ultimo procedemos a revisar cuantos registros tenemos de cada categoria.

In [34]:
travel_count = df[df['category'] == 'travel'].shape[0]
print("travel:", travel_count)

food_count = df[df['category'] == 'food'].shape[0]
print("food:", food_count)

art_music_count = df[df['category'] == 'art_music'].shape[0]
print("art_music:", art_music_count)

history_count = df[df['category'] == 'history'].shape[0]
print("history:", history_count)

travel: 1127
food: 887
art_music: 831
history: 588


Como podemos ver nuestras categorias no estan balanceadas, siendo mas las de 'travel' que son casi el doble de las de 'history'

Longitud de caracteres por titulo y descripcion

In [35]:
title_lengths = df['title'].str.len()
print("Titulo")
print("Promedio:", title_lengths.mean())
print("Mínimo:", title_lengths.min())
print("Máximo:", title_lengths.max())


Titulo
Promedio: 67.04340227206525
Mínimo: 6
Máximo: 101


In [36]:
title_lengths = df['description'].str.len()
print("Descripción")
print("Promedio:", title_lengths.mean())
print("Mínimo:", title_lengths.min())
print("Máximo:", title_lengths.max())

Descripción
Promedio: 426.7707544421788
Mínimo: 23
Máximo: 5239


Debido a la longitud de las descripciones optaremos por usar titulos para predecir la categoria.

# Modelo

In [37]:
from datasets import Dataset, ClassLabel
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
import evaluate

## Separar (train/val/test)

Crearemos una nueva columna llamada "Label" el cual tendra un valor numerico segun la categoria para facilitar la clasificación.  

A su vez aprovechamos para separar nuestros datos en entrenamiento y validacion.

In [38]:
classes = sorted(df["category"].astype(str).unique().tolist())
label2id = {c:i for i, c in enumerate(classes)}
id2label = {i:c for c, i in label2id.items()}

# Crear columna 'label' numérica a partir de 'category' para clasificación
df = df.copy()
df["label"] = df["category"].astype(str).map(label2id)

# Split estratificado
train_df, val_df = train_test_split(
    df[["title","label"]],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)

### RoBERTa

Se eligio Roberta por que tiene mejores resultados que BERT ya que fue entrenado con más datos por más tiempo, permite ajustar el modelo con pocas etiquetas, ademas de la flexibilidad del modelo para trabajar con diferentes idiomas.

Usamos una longitud maxima de 150 caracteres para los titulos ya que el maximo es 101 y prometio de 67.  
Generamos los tokens por palabra y revisamos las labels de nuestros dos sets de datos.

In [39]:
MODEL_NAME = "roberta-base"
MAX_LEN = 150

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess(batch):
    return tokenizer(
        batch["title"], truncation=True, padding=False, max_length=MAX_LEN
    )

train_ds = train_ds.map(preprocess, batched=True)
val_ds   = val_ds.map(preprocess,   batched=True)

# Trainer espera la columna 'labels'
train_ds = train_ds.rename_columns({"label": "labels"})
val_ds   = val_ds.rename_columns({"label": "labels"})

data_collator = DataCollatorWithPadding(tokenizer)


Map: 100%|██████████| 687/687 [00:00<00:00, 42751.40 examples/s]


### Modelo

Asignamos el numero de labels segun nuestras clases unicas (4) y apartir del modelo pre entrenado le pasamos los parametros corresponientes.

In [40]:
num_labels = len(classes)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Entrenamiento/ Evaluacion

Para las metricas usaremos accuracy (Que tantas veces acierta) y F1 Score (Entre mas cercano a 1 muestra más valores correctamente predichos).  
Es importante mencionar tambien la configuracion de nuestro entrenamiento segun epocas con un learning rate valido.  
Al final guardamos nuestro modelo para tener reproducibilidad mas adelante sin tener que correr de nuevo el modelo.

In [41]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

args = TrainingArguments(
    output_dir="roberta-cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
print(trainer.evaluate())
trainer.save_model("roberta-cls/best")
tokenizer.save_pretrained("roberta-cls/best")

C:\Users\death\AppData\Local\Temp\ipykernel_21392\487552205.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.124880,0.965066,0.967569
2,No log,0.084157,0.978166,0.979575
3,0.176500,0.120014,0.973799,0.975419


C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.08415716141462326, 'eval_accuracy': 0.9781659388646288, 'eval_f1_macro': 0.9795749148723972, 'eval_runtime': 25.968, 'eval_samples_per_second': 26.456, 'eval_steps_per_second': 1.656, 'epoch': 3.0}


('roberta-cls/best\\tokenizer_config.json',
 'roberta-cls/best\\special_tokens_map.json',
 'roberta-cls/best\\vocab.json',
 'roberta-cls/best\\merges.txt',
 'roberta-cls/best\\added_tokens.json',
 'roberta-cls/best\\tokenizer.json')

Como podemos ver tanto nuestro Accuracy como nuestro F1 score muestran resultados de 0.97 lo cual es muy bueno ya que casi todo el tiempo predice de manera correcta.  

Sin embargo hay que comprobar esto con el siguiente bloque.

In [42]:
import evaluate
import torch

def predict(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        out = model(**{k: v.to(model.device) for k, v in enc.items()})
        pred_ids = out.logits.argmax(dim=-1).cpu().numpy().tolist()
    return [model.config.id2label[i] for i in pred_ids]

print(predict([
    "I loved the museum and the old town tour.",
    "The tacos were amazing and fresh!",
    "El concierto de anoche fue espectacular, la banda tocó todos sus éxitos y el público estaba encantado."
]))

['travel', 'food', 'art_music']


Al pasarle una titulo o un texto lo puede clasificar en nuestras categorias esperadas sin problemas, sin embargo recordemos que esta entrenado con un conjunto de titulos ya etiquetados por lo cual puede que en algunos casos la interpretacion propia de cada persona pueda ser diferente.

# Cargar y Evaluar Modelo ya Entrenado

Para evitar entrenar el modelo cada vez cargamos nuestro modelo ya entrenado para poder comprobar directamente.

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MAX_LEN = 256
model = AutoModelForSequenceClassification.from_pretrained("roberta-cls/best")
tokenizer = AutoTokenizer.from_pretrained("roberta-cls/best")

def predict(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        out = model(**{k: v.to(model.device) for k, v in enc.items()})
        pred_ids = out.logits.argmax(dim=-1).cpu().numpy().tolist()
    return [model.config.id2label[i] for i in pred_ids]

# Ahora prueba
print(predict(["We went on a long flight to Paris.", "The pizza was delicious."]))


c:\Program Files\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['travel', 'food']
